In [12]:
# ChromBERT region-region interaction Inference Example
#
# This notebook demonstrates how to infer region-region interactions
# using the ChromBERT-tools Python API.

from chrombert_tools import interpret_region_region_interactions

In [13]:
help(interpret_region_region_interactions)

Help on function interpret_region_region_interactions in module chrombert_tools.api.interpret_region_region_interactions:

interpret_region_region_interactions(region: 'str', odir: 'str' = './output', genome: 'str' = 'hg38', resolution: 'str' = '1kb', chrombert_cache_dir: 'Optional[str]' = None, chrombert_region_file: 'Optional[str]' = None, chrombert_region_emb_file: 'Optional[str]' = None, return_data: 'bool' = True, region2: 'Optional[str]' = None, distance_window: 'int' = 250000, batch_size: 'int' = 4, ft_ckpt: 'Optional[str]' = None, ignore_regulator: 'Optional[str]' = None, gep: 'bool' = False, flank_window: 'int' = 4, chrombert_gene_meta: 'Optional[str]' = None) -> 'Optional[pd.DataFrame]'
    Compute cosine similarities between region embeddings: either enhancer-promoter
    style pairs (single ``region`` BED vs gene TSS from metadata) or all pairs
    between two BEDs on the same chromosome within ``distance_window``.
    
    Args:
        region:
            Path to BED file

### infer region-region interactions (enhancer-promoter loop; only by pretrained chrombert)

In [3]:
import pandas as pd
!mkdir -p 'output_infer_ep'
data = pd.read_csv("../data/hESC_GSM2386582_ATAC.bed",sep='\t',header=None)
data
data[data[0]=='chr1'].to_csv('./output_infer_ep/hESC_chr1.bed',sep='\t',header=False,index=False)

In [ ]:
# Returns:
# tss_region_pairs_cos: cosine similarity matrix of region–region representations; indicates interaction strength between enhancer and promoter regions.


tss_region_pairs_cos = interpret_region_region_interactions(
    region='./output_infer_ep/hESC_chr1.bed', # your focus enhancer region
    odir="./output_infer_ep", # output directory
    genome="hg38",                      # Options: "hg38", "mm10"
    resolution="1kb",                   # Options: "1kb", "2kb", "4kb", "200bp"
)

Region summary - total: 5262, overlapping with ChromBERT: 5490 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 33
Finished!
Enhancer-promoter style pairs saved to: ./output_infer_ep/tss_region_pairs_cos.tsv


In [5]:
tss_region_pairs_cos.head()

,chrom,gene_id,gene_name,tss,tss_build_region_index,distal_region_start,distal_region_end,distal_region_build_region_index,dist,dist_bin,cos_sim
0,chr1,ENSG00000278267,MIR6859-1,17436,2,10000,11000,0,-6436,-2,0.609375
1,chr1,ENSG00000278267,MIR6859-1,17436,2,181000,182000,39,163564,37,0.738281
2,chr1,ENSG00000227232,WASH7P,29570,3,10000,11000,0,-18570,-3,0.599121
3,chr1,ENSG00000227232,WASH7P,29570,3,181000,182000,39,151430,36,0.706543
4,chr1,ENSG00000243485,MIR1302-2HG,29554,3,10000,11000,0,-18554,-3,0.599121


In [6]:
tss_region_pairs_cos.query("gene_name == 'RNVU1-15'").sort_values(by='cos_sim',ascending=False)

,chrom,gene_id,gene_name,tss,tss_build_region_index,distal_region_start,distal_region_end,distal_region_build_region_index,dist,dist_bin,cos_sim
47104,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144546000,144547000,99004,133424,79,0.966797
47105,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144551000,144552000,99008,138424,83,0.910645
47106,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144560000,144561000,99015,147424,90,0.794922
47101,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144461000,144462000,98949,48424,24,0.699219
47100,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144419000,144420000,98930,6424,5,0.688477
47103,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144524000,144525000,98985,111424,60,0.651367
47107,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144567000,144568000,99021,154424,96,0.431885
47102,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144490000,144491000,98966,77424,41,0.307373


### infer enhancer-promoter interactions (celltype-specific fine-tuned model)

In [14]:
import pandas as pd
!mkdir -p 'output_infer_ep_myoblast_specific'
data = pd.read_csv("../data/myoblast_ENCFF647RNC_peak.bed",sep='\t',header=None)
data
data[data[0]=='chr1'].to_csv('./output_infer_ep_myoblast_specific/myoblast_chr1.bed',sep='\t',header=False,index=False)

In [15]:
import glob
ft_ckpt_dir = "./output_cell_specific_emb_cistrome/train/**/*.ckpt"

ft_ckpt = glob.glob(ft_ckpt_dir, recursive=True)[0]
ft_ckpt

'./output_cell_specific_emb_cistrome/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=176.ckpt'

In [17]:
# Returns:
# tss_region_pairs_cos: cosine similarity matrix of region–region representations; indicates interaction strength between enhancer and promoter regions.


tss_region_pairs_cos = interpret_region_region_interactions(
    region='./output_infer_ep_myoblast_specific/myoblast_chr1.bed', # your focus enhancer region
    odir="./output_infer_ep_myoblast_specific", # output directory
    genome="hg38",                      # Options: "hg38", "mm10"
    resolution="1kb",                   # Options: "1kb", "2kb", "4kb", "200bp"
    ft_ckpt=ft_ckpt,
    batch_size=64,
)

Region summary - total: 33691, overlapping with ChromBERT: 33479 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 482
Your supervised_file does not contain the 'label' column. Please verify whether ground truth column ('label') is required. If it is not needed, you may disregard this message.
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Loading checkpoint from ./output_cell_specific_emb_cistrome/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=176.ckpt
Loading from pl module, remove prefix 'model.'
Loading from pl module, replace 'pretrain_model' with 'pretrain_model.chrombert'
Loaded 111/111 parameters


100%|██████████| 1222/1222 [20:32<00:00,  1.01s/it]


Finished!
Enhancer-promoter style pairs saved to: ./output_infer_ep_myoblast_specific/tss_region_pairs_cos.tsv


In [22]:
tss_region_pairs_cos.query("gene_name == 'RNVU1-15'").sort_values(by='cos_sim',ascending=False)

,chrom,gene_id,gene_name,tss,tss_build_region_index,distal_region_start,distal_region_end,distal_region_build_region_index,dist,dist_bin,cos_sim
242161,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144546000,144547000,99004,133424,79,0.974380
242160,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144546000,144547000,99004,133424,79,0.974380
242165,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144552000,144553000,99009,139424,84,0.949564
242164,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144551000,144552000,99008,138424,83,0.934053
242166,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144560000,144561000,99015,147424,90,0.799996
242158,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144524000,144525000,98985,111424,60,0.676938
242138,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144413000,144414000,98926,424,1,0.670687
242163,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144550000,144551000,99007,137424,82,0.632238
242139,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144419000,144420000,98930,6424,5,0.625515
242140,chr1,ENSG00000207205,RNVU1-15,144412576,98925,144419000,144420000,98930,6424,5,0.625515


### infer region-region interactions (two region groups)

In [7]:
from chrombert_tools import resolve_paths
chrombert_anno_files = resolve_paths(genome="hg38", resolution="1kb",chrombert_cache_dir="~/.cache/chrombert/data")

In [8]:
chrombert_anno_files

{'chrombert_cache_dir': '/mnt/Storage/home/chenqianqian/.cache/chrombert/data',
 'genome': 'hg38',
 'resolution': '1kb',
 'chrombert_region_file': '/mnt/Storage/home/chenqianqian/.cache/chrombert/data/config/hg38_6k_1kb_region.bed',
 'chrombert_regulator_file': '/mnt/Storage/home/chenqianqian/.cache/chrombert/data/config/hg38_6k_regulators_list.txt',
 'chrombert_factor_file': '/mnt/Storage/home/chenqianqian/.cache/chrombert/data/config/hg38_6k_factors_list.txt',
 'hdf5_file': '/mnt/Storage/home/chenqianqian/.cache/chrombert/data/hg38_6k_1kb.hdf5',
 'pretrain_ckpt': '/mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt',
 'mtx_mask': '/mnt/Storage/home/chenqianqian/.cache/chrombert/data/config/hg38_6k_mask_matrix.tsv',
 'region_emb_npy': '/mnt/Storage/home/chenqianqian/.cache/chrombert/data/anno/hg38_1kb_region_emb.npy',
 'gene_meta_tsv': '/mnt/Storage/home/chenqianqian/.cache/chrombert/data/anno/hg38_1kb_gene_meta.tsv',
 'base_ca_signal': '/mnt/Stor

In [9]:
import pandas as pd
gene_meta = pd.read_csv(chrombert_anno_files["gene_meta_tsv"],sep='\t')
gene_meta.head()

,chrom,loc1,loc2,strand,tss,gene_id,gene_name,gene_biotype,start,end,build_region_index
0,chr1,182696,184174,+,182696,ENSG00000279928,DDX11L17,unprocessed_pseudogene,182000,183000,40
1,chr1,2581560,2584533,+,2581560,ENSG00000228037,NaN,lncRNA,2581000,2582000,1762
2,chr1,3069168,3438621,+,3069168,ENSG00000142611,PRDM16,protein_coding,3069000,3070000,2125
3,chr1,5301928,5307394,-,5307394,ENSG00000284616,NaN,lncRNA,5307000,5308000,3910
4,chr1,2403964,2413797,-,2413797,ENSG00000157911,PEX10,protein_coding,2413000,2414000,1600


In [ ]:
# Returns:
# tss_region_pairs_cos: cosine similarity matrix of region–region representations; indicates interaction strength between enhancer and promoter regions.


tss_region_pairs_cos = interpret_region_region_interactions(
    region='./output_infer_ep/hESC_chr1.bed', # your focus region group 1
    region2=chrombert_anno_files["gene_meta_tsv"], # your focus region group 2
    odir="./output_region_region_interactions", # output directory
    genome="hg38",                      # Options: "hg38", "mm10"
    resolution="1kb",                   # Options: "1kb", "2kb", "4kb", "200bp"
)

Region summary - total: 5262, overlapping with ChromBERT: 5490 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 33
Region summary - total: 55240, overlapping with ChromBERT: 55240 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 0
Finished!
Set1 x set2 region-pair cosines (same chrom, genomic_dist_bp <= 250000) saved to: ./output_region_region_interactions/region_set_pairs_cos.tsv


In [11]:
tss_region_pairs_cos

,set1_chrom,set1_start,set1_end,set1_build_region_index,set2_chrom,set2_start,set2_end,set2_build_region_index,genomic_dist_bp,cos_sim
0,chr1,10000,11000,0,chr1,17000,18000,2,6000,0.609604
1,chr1,10000,11000,0,chr1,29000,30000,3,18000,0.599131
2,chr1,10000,11000,0,chr1,29000,30000,3,18000,0.599131
3,chr1,10000,11000,0,chr1,30000,31000,4,19000,0.604889
4,chr1,10000,11000,0,chr1,91000,92000,15,80000,0.667273
...,...,...,...,...,...,...,...,...,...,...
86022,chr1,248946000,248947000,183982,chr1,248838000,248839000,183892,107000,0.168999
86023,chr1,248946000,248947000,183982,chr1,248859000,248860000,183909,86000,0.176858
86024,chr1,248946000,248947000,183982,chr1,248859000,248860000,183909,86000,0.176858
86025,chr1,248946000,248947000,183982,chr1,248906000,248907000,183953,39000,0.206660
